# Export figures as SVG

Generates plots and saves each as a vector SVG into `results/figures/vector`. Trimmed from `Sample_comparison_IHOPE.ipynb` down to just the setup needed for these figures.

Covers: heatmaps (sample clustermap + tissue heatmap), stacked barplots (type-level + CD4/CD8 T cell state splits), strip plots (all cell types), and fold change plots.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt

%config InlineBackend.close_figures = False

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

VECTOR_DIR = Path("/Users/emmyberg/Documents/IHOPE_SpatialProteomics/results/figures/vector")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import importlib
from scripts import comparison
importlib.reload(comparison)

from scripts.comparison import (
    load_celltype_summaries,
    pivot_for_heatmap,
    plot_celltype_heatmap,
    pivot_for_tissue_heatmap,
    plot_celltype_clustermap,
    plot_celltype_stacked_barplot,
    plot_celltype_stripplot,
    normalise_to_parent,
    parent_ratio_labels,
)

from scripts import differential_screen as ds
importlib.reload(ds)


### Load data

In [ ]:
name_map = {
    "celltype_summary_IHOPE14_MedLN_BottomLeft.csv": "IHOPE14 MedLN Bottom Left",
    "celltype_summary_IHOPE14_MedLN_BottomRight.csv": "IHOPE14 MedLN Bottom Right",
    "celltype_summary_IHOPE14_MedLN_TopRight.csv": "IHOPE14 MedLN Top Right",
    "celltype_summary_IHOPE14_MesLN.csv": "IHOPE14 MesLN",
    "celltype_summary_IHOPE20_MedLN.csv": "IHOPE20 MedLN",
    "celltype_summary_IHOPE20_Spleen.csv": "IHOPE20 Spleen",
    "celltype_summary_IHOPE26_MedLN.csv": "IHOPE26 MedLN",
    "celltype_summary_IHOPE26_Spleen.csv": "IHOPE26 Spleen",
    "celltype_summary_IHOPE27_MedLN.csv": "IHOPE27 MedLN",
    "celltype_summary_IHOPE27_Spleen.csv": "IHOPE27 Spleen",
    "celltype_summary_IHOPE39_MedLN.csv": "IHOPE39 MedLN",
    "celltype_summary_IHOPE39_MesLN_A.csv": "IHOPE39 MesLN A",
    "celltype_summary_IHOPE39_MesLN_B.csv": "IHOPE39 MesLN B",
    "celltype_summary_IHOPE39_Spleen.csv": "IHOPE39 Spleen",
}

summaries_dir = Path("../results/reports/zscore_log2")


In [ ]:
file_map = {
    sample: summaries_dir / fname
    for fname, sample in name_map.items()
}


In [ ]:
df = load_celltype_summaries(file_map=file_map)


### Donor grouping

In [ ]:
donor_map = {
    "IHOPE14 MedLN Bottom Left": "IHOPE14",
    "IHOPE14 MedLN Bottom Right": "IHOPE14",
    "IHOPE14 MedLN Top Right": "IHOPE14",
    "IHOPE14 MesLN": "IHOPE14",
    "IHOPE20 MedLN": "IHOPE20",
    "IHOPE20 Spleen": "IHOPE20",
    "IHOPE26 MedLN": "IHOPE26",
    "IHOPE26 Spleen": "IHOPE26",
    "IHOPE27 MedLN": "IHOPE27",
    "IHOPE27 Spleen": "IHOPE27",
    "IHOPE39 MedLN": "IHOPE39",
    "IHOPE39 MesLN A": "IHOPE39",
    "IHOPE39 MesLN B": "IHOPE39",
    "IHOPE39 Spleen": "IHOPE39",
}

df["donor"] = df["sample"].map(donor_map)


In [ ]:
donor_ids = sorted(set(donor_map.values()))
donor_colors = dict(zip(donor_ids, sns.color_palette("colorblind", n_colors=len(donor_ids))))


### Tissue grouping

In [ ]:
tissue_map = {
    "IHOPE14 MedLN Bottom Left": "MedLN",
    "IHOPE14 MedLN Bottom Right": "MedLN",
    "IHOPE14 MedLN Top Right": "MedLN",
    "IHOPE14 MesLN": "MesLN",
    "IHOPE20 MedLN": "MedLN",
    "IHOPE20 Spleen": "Spleen",
    "IHOPE26 MedLN": "MedLN",
    "IHOPE26 Spleen": "Spleen",
    "IHOPE27 MedLN": "MedLN",
    "IHOPE27 Spleen": "Spleen",
    "IHOPE39 MedLN": "MedLN",
    "IHOPE39 MesLN A": "MesLN",
    "IHOPE39 MesLN B": "MesLN",
    "IHOPE39 Spleen": "Spleen",
}

df["tissue"] = df["sample"].map(tissue_map)


In [ ]:
df_tissue = (
    df.groupby(["tissue", "level", "cell_type"], as_index=False)
      .agg(pct_total=("pct_total", "mean"))
)


In [ ]:
DROP_UNCLASSIFIED = True   # drop 'unclassified' and rescale to 100, applied in the matrix and barplot calls below
HIDE_UNCLASSIFIED = False  # keep percentages as-is, just drop the unclassified row from heatmaps


In [ ]:
# Structural / non-immune cell types, dropped when immune_only=True
structural_cell_types = [
    "Blood_Endothelial",
    "Lymphatic_Endothelial",
    "Basement_Membrane",
    "Fibroblast",
    "Stromal",
    "Endothelial",
    "FDC"
]

# Lineage subsets for the per-lineage breakdown barplots
lineage_subsets = {
    "T": [
        "Activated_CD4", "Activated_CD8", "TCM_CD4", "TCM_CD8",
        "TEM_CD4", "TEM_CD8", "TEMRA_CD4", "TEMRA_CD8",
        "TN_CD4", "TN_CD8", "Treg", "TfH_like", "T_terminal",
    ],
    "B": [
        "B_naive", "B_GC", "B_Plasmablast",
    ],
    "Myeloid": [
        "Monocyte_Macrophage", "cDC1", "cDC2",
    ],
}

# Manual row order for heatmaps, grouping related subtypes together.
celltype_order = [
    "T", "NK", "B", "Myeloid", "Stromal", "Endothelial", "unclassified",
    "T_naive", "T_memory", "CD4_T", "CD8_T", "B_memory",
    "TN_CD4", "TCM_CD4", "TEM_CD4", "TEMRA_CD4", "Activated_CD4", "Treg", "TfH_like",
    "TN_CD8", "TCM_CD8", "TEM_CD8", "TEMRA_CD8", "Activated_CD8",
    "T_terminal",
    "B_naive", "B_GC", "B_Plasmablast",
    "Monocyte_Macrophage", "cDC1", "cDC2",
    "FDC", "Fibroblast", "Basement_Membrane",
    "Blood_Endothelial", "Lymphatic_Endothelial",
]

# Parent population for each row in the parent-relative heatmaps.
parent_of = {
    "CD4_T": "T", "CD8_T": "T", "T_naive": "T", "T_memory": "T", "T_terminal": "T",
    "TN_CD4": "CD4_T", "TCM_CD4": "CD4_T", "TEM_CD4": "CD4_T", "TEMRA_CD4": "CD4_T",
    "Activated_CD4": "CD4_T", "Treg": "CD4_T", "TfH_like": "CD4_T",
    "TN_CD8": "CD8_T", "TCM_CD8": "CD8_T", "TEM_CD8": "CD8_T",
    "TEMRA_CD8": "CD8_T", "Activated_CD8": "CD8_T",
    "B_naive": "B", "B_memory": "B", "B_GC": "B", "B_Plasmablast": "B",
    "Monocyte_Macrophage": "Myeloid", "cDC1": "Myeloid", "cDC2": "Myeloid",
}

# Root level, no single "Immune" row exists so the denominator is the sum
# of these type-level rows.
immune_types = ["T", "B", "NK", "Myeloid"]

# Display names for the parent-relative heatmap row labels.
celltype_display = {
    "T": "T cells", "B": "B cells", "NK": "NK cells", "Myeloid": "Myeloid cells",
    "CD4_T": "CD4 T cells", "CD8_T": "CD8 T cells",
    "T_naive": "Naive T cells", "T_memory": "Memory T cells",
    "T_terminal": "Terminally differentiated T cells",
    "TN_CD4": "Naive CD4 T cells", "TCM_CD4": "TCM CD4 T cells",
    "TEM_CD4": "TEM CD4 T cells", "TEMRA_CD4": "TEMRA CD4 T cells",
    "Activated_CD4": "Activated CD4 T cells", "Treg": "Regulatory T cells",
    "TfH_like": "TfH-like cells",
    "TN_CD8": "Naive CD8 T cells", "TCM_CD8": "TCM CD8 T cells",
    "TEM_CD8": "TEM CD8 T cells", "TEMRA_CD8": "TEMRA CD8 T cells",
    "Activated_CD8": "Activated CD8 T cells",
    "B_naive": "Naive B cells", "B_memory": "Memory B cells",
    "B_GC": "GC B cells", "B_Plasmablast": "Plasmablasts",
    "Monocyte_Macrophage": "Monocytes and macrophages",
    "cDC1": "cDC1", "cDC2": "cDC2",
}


In [ ]:
tissue_order = ["MedLN", "MesLN", "Spleen"]


In [ ]:
# Color palette for cell types
celltype_colors = {
    "T": "#d62728", "B": "#1f77b4", "NK": "#2ca02c", "Myeloid": "#e377c2",
    "Stromal": "#8c564b", "Endothelial": "#9467bd", "unclassified": "#d9d9d9",
    "CD4_T": "#d62728", "CD8_T": "#ff7f0e", "T_naive": "#9467bd",
    "T_memory": "#2ca02c", "B_memory": "#1f77b4",
    "B_naive": "#aec7e8", "B_GC": "#1f77b4", "B_Plasmablast": "#17becf",
    "TN_CD4": "#d62728", "TCM_CD4": "#ff7f0e", "TEM_CD4": "#ff9896",
    "TEMRA_CD4": "#c5b0d5", "Activated_CD4": "#ad494a", "Treg": "#843c39",
    "TfH_like": "#e7298a",
    "TN_CD8": "#fdd0a2", "TCM_CD8": "#ffbb78", "TEM_CD8": "#bcbd22",
    "TEMRA_CD8": "#dbdb8d", "Activated_CD8": "#8c6d31",
    "T_terminal": "#7f7f7f",
    "cDC1": "#2ca02c", "cDC2": "#98df8a", "Monocyte_Macrophage": "#8c564b",
    "FDC": "#9467bd", "Fibroblast": "#c49c94", "Basement_Membrane": "#c7c7c7",
    "Blood_Endothelial": "#393b79", "Lymphatic_Endothelial": "#5254a3",
}


### Cell type matrices

In [ ]:
matrix_all = pivot_for_heatmap(
    df, celltype_order=celltype_order, tissue_order=tissue_order,
    drop_unclassified=DROP_UNCLASSIFIED, renormalise=DROP_UNCLASSIFIED,
)
matrix_immune = pivot_for_heatmap(
    df, immune_only=True, renormalise=True,
    structural_cell_types=structural_cell_types,
    celltype_order=celltype_order, tissue_order=tissue_order,
)

matrix_tissue_all = pivot_for_tissue_heatmap(
    df_tissue, celltype_order=celltype_order,
    drop_unclassified=DROP_UNCLASSIFIED, renormalise=DROP_UNCLASSIFIED,
)
matrix_tissue_immune = pivot_for_tissue_heatmap(
    df_tissue, immune_only=True, renormalise=True,
    structural_cell_types=structural_cell_types,
    celltype_order=celltype_order,
)

matrix_parent = normalise_to_parent(
    pivot_for_heatmap(
        df, celltype_order=celltype_order, tissue_order=tissue_order,
        drop_unassigned=True,
    ),
    parent_of, immune_types,
)
matrix_tissue_parent = normalise_to_parent(
    pivot_for_tissue_heatmap(
        df_tissue, celltype_order=celltype_order, drop_unassigned=True,
    ),
    parent_of, immune_types,
)


## Plots

In [ ]:
USE_LOG = False           # log scaling, ignored when HEATMAP_NORM == "parent"
HEATMAP_NORM = "parent"   # "all", "immune", or "parent"
IMMUNE_ONLY = True        # used by the stacked barplots below, not by the heatmaps

if HEATMAP_NORM == "parent":
    matrix = matrix_parent
elif HEATMAP_NORM == "immune":
    matrix = matrix_immune
else:
    matrix = matrix_all

use_log = USE_LOG and HEATMAP_NORM != "parent"
scale_label = "log" if use_log else "linear"

if HEATMAP_NORM == "parent":
    cbar_label = "Percentage"
    heatmap_vmax = 100
else:
    cbar_label = "Log percentage (+0.1)" if use_log else "Percentage"
    heatmap_vmax = None

plot_matrix = np.log10(matrix + 0.1) if use_log else matrix

if HIDE_UNCLASSIFIED and HEATMAP_NORM != "parent":
    plot_matrix = plot_matrix.drop(index="unclassified", errors="ignore")

plot_matrix = plot_matrix.fillna(0)

if HEATMAP_NORM == "parent":
    plot_matrix = parent_ratio_labels(
        plot_matrix, parent_of, immune_types, display_names=celltype_display,
    )


**Heatmap, clustered (correlation metric), sample-level**

In [ ]:
plot_celltype_clustermap(
    plot_matrix,
    scale=scale_label,
    colorbar_legend=cbar_label,
    cluster_rows=False,
    cluster_cols=True,
    metric="correlation",
    vmax=heatmap_vmax,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "heatmap_clustered_sample.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
USE_LOG = False   # log scaling, ignored when HEATMAP_NORM == "parent"

if HEATMAP_NORM == "parent":
    matrix_tissue = matrix_tissue_parent
elif HEATMAP_NORM == "immune":
    matrix_tissue = matrix_tissue_immune
else:
    matrix_tissue = matrix_tissue_all

use_log_tissue = USE_LOG and HEATMAP_NORM != "parent"
scale_label = "log" if use_log_tissue else "linear"

if HEATMAP_NORM == "parent":
    cbar_label = "Percentage"
    heatmap_vmax_tissue = 100
else:
    cbar_label = "Percentage (log +0.1)" if use_log_tissue else "Percentage"
    heatmap_vmax_tissue = None

plot_matrix_tissue = np.log10(matrix_tissue + 0.1) if use_log_tissue else matrix_tissue

if HIDE_UNCLASSIFIED and HEATMAP_NORM != "parent":
    plot_matrix_tissue = plot_matrix_tissue.drop(index="unclassified", errors="ignore")

plot_matrix_tissue = plot_matrix_tissue.fillna(0)

if HEATMAP_NORM == "parent":
    plot_matrix_tissue = parent_ratio_labels(
        plot_matrix_tissue, parent_of, immune_types, display_names=celltype_display,
    )


**Heatmap, tissue-level**

In [ ]:
plot_celltype_heatmap(
    plot_matrix_tissue,
    colorbar_legend=cbar_label,
    scale=scale_label,
    vmax=heatmap_vmax_tissue,
    narrow=True,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "heatmap_tissue.svg", bbox_inches="tight")
plt.close(fig)


### Stacked barplots

In [ ]:
LN_ONLY = False  # set True to restrict barplots to lymph node samples only (MedLN + MesLN, drop Spleen)

data_sample = df[df["tissue"] != "Spleen"] if LN_ONLY else df
data_tissue = df_tissue[df_tissue["tissue"] != "Spleen"] if LN_ONLY else df_tissue


**Type-level**

In [ ]:
palette_type = plot_celltype_stacked_barplot(
    data_sample,
    levels=["type"],
    immune_only=IMMUNE_ONLY,
    structural_cell_types=structural_cell_types,
    drop_unclassified=DROP_UNCLASSIFIED,
    tissue_order=tissue_order,
    ylim=(0, 100),
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "type_barplot_sample.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
plot_celltype_stacked_barplot(
    data_sample,
    levels=["type"],
    x="tissue",
    palette=palette_type,
    immune_only=IMMUNE_ONLY,
    structural_cell_types=structural_cell_types,
    drop_unclassified=DROP_UNCLASSIFIED,
    ylim=(0, 100),
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "type_barplot_tissue.svg", bbox_inches="tight")
plt.close(fig)


**T cell state splits, CD4/CD8**

In [ ]:
lineage_subsets["CD4_T"] = ["TN_CD4", "TCM_CD4", "TEM_CD4", "TEMRA_CD4"]
lineage_subsets["CD8_T"] = ["TN_CD8", "TCM_CD8", "TEM_CD8", "TEMRA_CD8"]


In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="sample",
    lineage_subset="CD4_T",
    lineage_subsets=lineage_subsets,
    title="CD4 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD4", "TN_CD4", "TEM_CD4", "TCM_CD4"],
    palette=celltype_colors,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "CD4_Tc_barplot_sample.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="tissue",
    lineage_subset="CD4_T",
    lineage_subsets=lineage_subsets,
    title="CD4 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD4", "TN_CD4", "TEM_CD4", "TCM_CD4"],
    palette=celltype_colors,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "CD4_Tc_barplot_tissue.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="sample",
    lineage_subset="CD8_T",
    lineage_subsets=lineage_subsets,
    title="CD8 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD8", "TN_CD8", "TEM_CD8", "TCM_CD8"],
    palette=celltype_colors,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "CD8_Tc_barplot_sample.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
plot_celltype_stacked_barplot(
    df,
    levels=["subtype"],
    x="tissue",
    lineage_subset="CD8_T",
    lineage_subsets=lineage_subsets,
    title="CD8 T cell states",
    tissue_order=tissue_order,
    celltype_order=["TEMRA_CD8", "TN_CD8", "TEM_CD8", "TCM_CD8"],
    palette=celltype_colors,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "CD8_Tc_barplot_tissue.svg", bbox_inches="tight")
plt.close(fig)


### Strip plots, intermediate and subtypes, relative to parent populations

In [ ]:
plot_celltype_stripplot(
    df,
    level="subtype",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    parent_of=parent_of,
    root_children=immune_types,
    display_names=celltype_display,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "stripplot_subtype_tissue.svg", bbox_inches="tight")
plt.close(fig)

In [ ]:
plot_celltype_stripplot(
    df,
    level="intermediate",
    x="tissue",
    tissue_order=tissue_order,
    donor_colors=donor_colors,
    parent_of=parent_of,
    root_children=immune_types,
    display_names=celltype_display,
)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "stripplot_intermediate_tissue.svg", bbox_inches="tight")
plt.close(fig)

### Fold change plots

Compares fold change (FC) and percentage point difference between MesLN and MedLN.

In [ ]:
from scripts import differential_screen as ds

reports_dir = "/Users/emmyberg/Documents/IHOPE_SpatialProteomics/results/reports/zscore_log2"
anndata_dir = "/Users/emmyberg/Documents/IHOPE_SpatialProteomics/data/anndata/zscore_log2/celltyped/follicledomains"

basenames = [
    "IHOPE14_MedLN_BottomLeft", "IHOPE14_MedLN_TopRight", "IHOPE14_MedLN_BottomRight",
    "IHOPE14_MesLN", "IHOPE20_MedLN", "IHOPE20_Spleen", "IHOPE26_MedLN",
    "IHOPE26_Spleen", "IHOPE27_MedLN", "IHOPE27_Spleen", "IHOPE39_MedLN",
    "IHOPE39_MesLN_A", "IHOPE39_MesLN_B", "IHOPE39_Spleen",
]

# cell type abundance screen
long_df = ds.load_celltype_long(reports_dir, basenames)
donor = ds.pool_to_donor_level(long_df)             # pct over all cells (raw, kept)
donor = ds.exclude_unresolved(donor)                # drops _unclassified and _unassigned, adds pct_resolved
donor = ds.add_parent_relative(donor, parent_of)    # pct_parent, invariant to the base

import importlib
from scripts import differential_screen as ds
importlib.reload(ds)

abund = ds.screen(donor, group_a="MesLN", group_b="MedLN", value_col="pct_resolved")
abund_parent = ds.screen(donor, group_a="MesLN", group_b="MedLN", value_col="pct_parent")
markers = ds.screen(marker_donor, group_a="MesLN", group_b="MedLN", value_col="pct")

# marker positivity screen
positivity_csv = "marker_positivity_by_sample.csv"
ds.extract_marker_positivity(anndata_dir, basenames, positivity_csv)
marker_donor = ds.marker_to_donor_level(positivity_csv)
markers = ds.screen(marker_donor, group_a="MesLN", group_b="MedLN", value_col="pct")

In [ ]:
ds.plot_fc_lollipop(abund, metric="log2fc", top_n=12, title="MesLN vs MedLN abundance, fold change")

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "abundance_foldchange.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
ds.plot_fc_lollipop(abund, metric="diff", top_n=12, title="MesLN vs MedLN abundance, percentage points")

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "abundance_diff.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
ds.plot_fc_lollipop(abund_parent, metric="log2fc", top_n=12, title="MesLN vs MedLN, parent-relative fold change")

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "abundance_parent_foldchange.svg", bbox_inches="tight")
plt.close(fig)


In [ ]:
ds.plot_fc_lollipop(markers, metric="log2fc", top_n=15, title="MesLN vs MedLN marker positivity, fold change")

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "marker_foldchange.svg", bbox_inches="tight")
plt.close(fig)


FC with individual donor values

In [ ]:
# with dot size scaled by abundance
ds.plot_fc_lollipop(abund, metric="log2fc", points="donor", size_by_abundance=True, top_n=12,
                    title="MesLN vs MedLN abundance, fold change per donor")
fig = plt.gcf()
fig.savefig(VECTOR_DIR / "abundance_foldchange_donor_sized.svg", bbox_inches="tight")
plt.close(fig)

ds.plot_fc_lollipop(abund_parent, metric="log2fc", points="donor", size_by_abundance=True, top_n=12,
                    title="MesLN vs MedLN, parent-relative fold change per donor")
fig = plt.gcf()
fig.savefig(VECTOR_DIR / "abundance_parent_foldchange_donor_sized.svg", bbox_inches="tight")
plt.close(fig)

ds.plot_fc_lollipop(markers, metric="log2fc", points="donor", size_by_abundance=True, top_n=15,
                    title="MesLN vs MedLN marker positivity per donor")
fig = plt.gcf()
fig.savefig(VECTOR_DIR / "marker_foldchange_donor_sized.svg", bbox_inches="tight")
plt.close(fig)

# same, without abundance sizing (uniform dots)
ds.plot_fc_lollipop(abund, metric="log2fc", points="donor", size_by_abundance=False, top_n=12,
                    title="MesLN vs MedLN abundance, fold change per donor")
fig = plt.gcf()
fig.savefig(VECTOR_DIR / "abundance_foldchange_donor.svg", bbox_inches="tight")
plt.close(fig)

ds.plot_fc_lollipop(abund_parent, metric="log2fc", points="donor", size_by_abundance=False, top_n=12,
                    title="MesLN vs MedLN, parent-relative fold change per donor")

### Follicle count

Boxplot with normalized follicle count (per 10 000 immune cells)

In [ ]:
FOLLICLE_OUT_DIR = Path("../results/reports/follicle_counts")

follicle_df = pd.read_csv(FOLLICLE_OUT_DIR / "follicle_counts_summary.csv")

follicle_tissue_map = {
    "IHOPE14_MedLN_BottomLeft": "MedLN",
    "IHOPE14_MedLN_BottomRight": "MedLN",
    "IHOPE14_MedLN_TopRight": "MedLN",
    "IHOPE20_MedLN": "MedLN",
    "IHOPE26_MedLN": "MedLN",
    "IHOPE27_MedLN": "MedLN",
    "IHOPE39_MedLN": "MedLN",
    "IHOPE14_MesLN": "MesLN",
    "IHOPE39_MesLN_A": "MesLN",
    "IHOPE39_MesLN_B": "MesLN",
    "IHOPE20_Spleen": "Spleen",
    "IHOPE26_Spleen": "Spleen",
    "IHOPE27_Spleen": "Spleen",
    "IHOPE39_Spleen": "Spleen",
}

follicle_donor_map = {
    "IHOPE14_MedLN_BottomLeft": "IHOPE14",
    "IHOPE14_MedLN_BottomRight": "IHOPE14",
    "IHOPE14_MedLN_TopRight": "IHOPE14",
    "IHOPE20_MedLN": "IHOPE20",
    "IHOPE26_MedLN": "IHOPE26",
    "IHOPE27_MedLN": "IHOPE27",
    "IHOPE39_MedLN": "IHOPE39",
    "IHOPE14_MesLN": "IHOPE14",
    "IHOPE39_MesLN_A": "IHOPE39",
    "IHOPE39_MesLN_B": "IHOPE39",
    "IHOPE20_Spleen": "IHOPE20",
    "IHOPE26_Spleen": "IHOPE26",
    "IHOPE27_Spleen": "IHOPE27",
    "IHOPE39_Spleen": "IHOPE39",
}

follicle_df["tissue"] = follicle_df["sample"].map(follicle_tissue_map)
follicle_df["donor"] = follicle_df["sample"].map(follicle_donor_map)

follicle_donor_ids = sorted(follicle_df["donor"].unique())
follicle_donor_colors = dict(zip(follicle_donor_ids, sns.color_palette("colorblind", n_colors=len(follicle_donor_ids))))

In [ ]:
plt.figure(figsize=(8, 6))

tissue_order_follicle = ["MedLN", "MesLN", "Spleen"]

sns.boxplot(
    data=follicle_df,
    x="tissue",
    y="follicles_per_10000_immune_cells",
    order=tissue_order_follicle,
    color="0.85",
    showfliers=False,
)

sns.stripplot(
    data=follicle_df,
    x="tissue",
    y="follicles_per_10000_immune_cells",
    order=tissue_order_follicle,
    hue="donor",
    palette=follicle_donor_colors,
    legend=False,
    jitter=0.15,
    s=6,
)

handles = [
    plt.Line2D([0], [0], marker="o", color="white", markerfacecolor=color,
               markersize=8, label=donor)
    for donor, color in follicle_donor_colors.items()
]
plt.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0),
           frameon=False, title="Donor")

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

plt.ylabel("Follicles per 10,000 immune cells")
plt.title("Normalized follicle counts by tissue")
plt.xlabel(None)
plt.tight_layout()
plt.grid(False)

fig = plt.gcf()
fig.savefig(VECTOR_DIR / "follicle_boxplot_tissue.svg", bbox_inches="tight")
plt.close(fig)